## EE 242 Lab 2b - Convolution - Denoising Audio Signals

**Hanlin Ma, Sparsh Dadhich, Amanda Zhang, Team AF07**

> **Curation note:** This notebook was reconstructed from an archived rendered notebook/PDF view because no local `.ipynb` file was available. A few long lines may remain clipped or compacted from that source.


In [ ]:
# We'll refer to this as the "import cell." Every module you import should be imported here.%matplotlib inlineimport numpy as npimport matplotlibimport scipy.signal as sigimport matplotlib.pyplot as plt# import whatever other modules you use in this lab -- there are more that you n[eed]from scipy.io.wavfile import read, writefrom IPython.display import Audioimport IPython.display as ipd

## SummaryIn this lab, you will work through a series of exercises to introduce you to working with audio signals and explore the impact of convolution and smoothing on the sound of the signals.## Assignment 2 -- Smoothing SignalsIn this assignment, we'll implement a moving window smoothing function to show how you can use convolution to remove noise from a signal. We'll use a discrete signal associated with a sampling period, and plot signals as if they were continuous to make it easier to see the effect of smoothing. The base signal is generated randomly, so you can run the cell multiple times to see how the results look for different signals. This assignment will have three parts, A-C.**A.** Using the starter code provided, create a base time signal and a noisy version of it by adding random noise generated with the numpy.random.randn() function (the standard normal distribution, which is zero mean and unit variance). Plot the original and noisy signals with 2x1 subplots, with the time axis labeled assuming a sampling rate of 1000 Hz. Constrain the y-axis to be [0,25] for all plots.**B.** Create a smoothed version of the signal called filtsig1 by computing the average value over a +/- k samples using the numpy.mean() function and k=20. You will need to make a decision as to how to handle the first and last k samples, for which there won't be a full k samples available in both directions. In a single plot, plot the noisy signal and the filtered signal overlaid on the original signal.**C.** Define a vector hfilt that corresponds to box of length N=2k+1 and height 1/N. Create a second smoothed version of the signal called filtsig2 by convolving the base signal with hfilt using the numpy.convolve() function. Plot the two different smoothed signal outputs overlaid on each other. Note that the convolve function will change the length, so you will need to define a new time vector for that.

In [ ]:
# Assignment 2 - Smoothing Signals# set up relevant parameterssrate = 1000  # sampling rate in Hztime = np.arange(0, 2, 1 / srate)  # associated time vector that corresponds to 2 se[conds]n = len(time)  # length of the time vector# here is a base signal to work with, values of signal points chosen randoml[y]p = 10   # points for piecewise linear signalamp = 20  # amplitude range of base signalbase = np.interp(np.linspace(0, p, n), np.arange(0, p), np.random.rand(p) * amp)# create some random noise to be added to the above base signalsnoiseamp = 2noise = noiseamp * np.random.randn(n)# Part A# Add noise to the base signals to create new noisy signals (this is just ad[dition])# TODO: Code that solves the rest of Anoise_added_base_signals = noise + baseplt.figure(figsize=(10, 6))# Plot original signalplt.subplot(2, 1, 1)plt.plot(time, base, label="Original Signal")plt.ylim([0, 25])plt.xlabel("Time (s)")plt.ylabel("Amplitude")plt.title("Original Signal")plt.legend()# Plot noisy signalplt.subplot(2, 1, 2)plt.plot(time, noise_added_base_signals, label="Noisy Signal")plt.ylim([0, 25])plt.xlabel("Time (s)")plt.ylabel("Amplitude")plt.title("Noisy Signal")plt.legend()plt.tight_layout()plt.show()# Part B# Implement the running mean filter with a for loop# For each sample, the output value at t = np.mean (x[t-k:t+k]).# Take care of border cases# TODO: Code that solves Bk = 20filtsig1 = np.copy(noise_added_base_signals)# # starting edge case (i = 0 to k-1)for i in range(0, k):    filtsig1[i] = np.mean(noise_added_base_signals[0:k+i+1])# middle case (k <= i <= n-k-1)for i in range(k, n-k):    filtsig1[i] = np.mean(noise_added_base_signals[i-k:i+k+1])# ending edge case (i = n-k to n-1)for i in range(n-k, n):    filtsig1[i] = np.mean(noise_added_base_signals[i-k:n])# Plot resultsplt.figure(figsize=(10, 4))plt.plot(time, base, label="Original Signal", alpha=0.7)plt.plot(time, noise_added_base_signals, label="Noisy Signal", alpha=0.5, li[newidth=1])plt.plot(time, filtsig1, label="Filtered Signal", linewidth=2)plt.ylim([0, 25])plt.xlabel("Time (s)")plt.ylabel("Amplitude")plt.title("Noisy and Filtered Signal (For Loop)")plt.legend()plt.show()# Part C# Implement smoothing using convolution# TODO: Code that solves CN = 2 * k + 1hfilt = np.ones(N) / Nfiltsig2 = np.convolve(noise_added_base_signals, hfilt, mode='same')plt.figure(figsize=(10, 4))plt.plot(time, base, label="Original Signal", alpha=0.7)plt.plot(time, noise_added_base_signals, label="Noisy Signal", alpha=0.5, li[newidth=1])plt.plot(time, filtsig2, label="Filtered Signal (Convolution)", linewidth=2)plt.ylim([0, 25])plt.xlabel("Time (s)")plt.ylabel("Amplitude")plt.title("Noisy and Filtered Signal (Convolution)")plt.legend()plt.grid()plt.show()

### DiscussionDescribe the differences in the results using the two methods and explain these differences in terms of system properties. Comment on how the results and plots change when you amplify the noise more and also change the value of k.The for-loop method explicitly manages edge cases so we can see that there isn't a downward signal on both edges like the convolution method have, the for-loop carefully averages the values by cases, whereas convolution applies an uniform filter across the entire signal. And the convolution method is also faster in terms of coding and operation. Increasing the noise amplitude result the noisy signal to fluctuate more widely around the base signal.Increasing k makes the filter smoother, which in cost loses more details in the original signal. A smaller k allows more noise to pass through, which reduces the effectiveness of the filter.

## Assignment 3 -- Removing Noise from an Audio SignalIn this assignment, we'll artificially add noise to an audio signal and then apply the system you used in Assignment 2 to remove it. You will need the audio packages that you used in Lab 1. This assignment will have three parts, A-C**A.** Read in the trombone sound file provided and name it tr_orig. This is a mono signal, so we will only need to worry about the single channel. Create a noise sequence that is the length of tr_orig using the numpy.random.randn() noise generation function with a scaling factor of 100. (It needs to be larger to be audible given the range of tr_orig.) Then add the two signals to create tr_noisy. Save tr_noisy to a new wav file.**B.** Apply the convolution filter from Assignment 2 to remove the noise from tr_noisy, creating a signal called tr_filt. Save this signal as a wav file.**C.** Read in both the noisy and filtered versions and play the two files and the original to hear the effect of the noise and noise removal. You may need to cast the values you write using wav.write into another data format because of a bug with scipy.

In [ ]:
# Assignment 3 - Removing Noise from an Audio Signal# Part A -  Create a noisy trombone signal.# First read the original trombone signal as tr_orig.# Create a noise signal like Assignment 2, and create the tr_noisy signal by[ adding them]# Save it as new wav file.# TODO: Code that solves Afile_name = 'trombone11.wav'sample_rate, tr_orig = read(file_name)tr_orig = tr_orig.astype('float64')print(f"Original Sample Rate: {sample_rate} Hz")print(f"tr_orig Shape: {tr_orig.shape}")  # ( Total number of sampling points,noiseamp = 100n = len(tr_orig)noise = noiseamp * np.random.randn(n)tr_noisy = tr_orig + noisewrite('tr_noisy.wav', sample_rate, tr_noisy.astype('int16'))# Part B - Filter the noisy signal# Apply the convolution filter you built in Assignment 2 Part C to filter th[e noisy signal]# Save it in a new wav file.# TODO: Code that solves Bk = 5N = 2 * k + 1hfilt = np.ones(N) / Ntr_filt = np.convolve(tr_noisy, hfilt, mode='same')write('tr_filt.wav', sample_rate, tr_filt.astype('int16'))# Part C - Observe the filtering capability# Play the files and hear the effects of adding noise and filtering.# TODO: Code that solves Cprint("Playing original file (trombone11.wav):")ipd.display(Audio("trombone11.wav"))print("Playing tr_noisy file (tr_noisy.wav):")ipd.display(Audio("tr_noisy.wav"))print("Playing tr_filt file (tr_filt.wav):")ipd.display(Audio("tr_filt.wav"))

### DiscussionComment on the differences in how the original and noise-removed signals sound. Comment on the impact of large increases or decreases in the value of k.The noisy signal have a very noticeable distortion a buzzing sound on top of the trombone audio, the filtered signal have a smoothed sound, but loses some of the sharp and treble details of the original signal, also has a lower volume. And by changing the k value, when increasing k, the filtered signal has an even lower volume and softer, which is the cost of removing a lots of noise from the signal.